# 04 Annotation Ceiling


## Method

This notebook estimates an experimental annotation ceiling by comparing labels from different UdonPred evaluation datasets on proteins that appear in both datasets. High agreement suggests that a model could, in principle, transfer between those disorder definitions. Low agreement suggests that the datasets encode different disorder concepts, contain annotation noise, or have limited overlap.

This is not model performance. It is an inter-annotation agreement estimate and should be interpreted as an approximate upper-bound diagnostic for cross-dataset transfer.


In [ ]:
import subprocess
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results"

ceiling_dir = RESULTS / "annotation_ceiling"
summary_csv = ceiling_dir / "annotation_ceiling_summary.csv"
overlap_csv = ceiling_dir / "overlap_details.csv"

if summary_csv.exists() and overlap_csv.exists():
    print("Annotation ceiling results already exist, skipping computation.")
else:
    print("Running annotation ceiling computation...")
    ceiling_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "estimate_annotation_ceiling.py"),
        "--udonpred-dir",
        str(ROOT / "UdonPred"),
        "--output-dir",
        str(ceiling_dir),
    ]
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    result.check_returncode()

ceiling_summary = pd.read_csv(summary_csv)
overlap_details = pd.read_csv(overlap_csv)

ceiling_summary["pair"] = ceiling_summary["dataset_a"] + " vs " + ceiling_summary["dataset_b"]
overlap_details["pair"] = overlap_details["dataset_a"] + " vs " + overlap_details["dataset_b"]

ceiling_summary.head()


In [ ]:
dataset_order = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]

all_pairs = ceiling_summary.drop_duplicates(["dataset_a", "dataset_b"])[
    ["dataset_a", "dataset_b", "match_mode", "n_proteins_overlap", "n_residues_compared"]
]

residue_overlap = pd.DataFrame(np.nan, index=dataset_order, columns=dataset_order)
protein_overlap = pd.DataFrame(np.nan, index=dataset_order, columns=dataset_order)
match_modes = pd.DataFrame("", index=dataset_order, columns=dataset_order)

for row in all_pairs.itertuples(index=False):
    residue_overlap.loc[row.dataset_a, row.dataset_b] = row.n_residues_compared
    residue_overlap.loc[row.dataset_b, row.dataset_a] = row.n_residues_compared
    protein_overlap.loc[row.dataset_a, row.dataset_b] = row.n_proteins_overlap
    protein_overlap.loc[row.dataset_b, row.dataset_a] = row.n_proteins_overlap
    match_modes.loc[row.dataset_a, row.dataset_b] = row.match_mode
    match_modes.loc[row.dataset_b, row.dataset_a] = row.match_mode

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    protein_overlap,
    annot=True,
    fmt=".0f",
    cmap="mako",
    mask=protein_overlap.isna(),
    ax=axes[0],
)
axes[0].set_title("Overlapping Proteins")
sns.heatmap(
    residue_overlap,
    annot=True,
    fmt=".0f",
    cmap="mako",
    mask=residue_overlap.isna(),
    ax=axes[1],
)
axes[1].set_title("Aligned Residues Compared")
plt.tight_layout()
plt.show()

all_pairs.sort_values("n_residues_compared", ascending=False).head(10)


## Overlap Caution

Agreement estimates are only meaningful when enough residues overlap. Pairs with very few overlapping residues should be interpreted cautiously, even if their agreement metric is high.


In [ ]:
small_overlap_pairs = all_pairs.sort_values("n_residues_compared").head(10)
small_overlap_pairs


## Primary Agreement Metric

For continuous-continuous dataset pairs, the primary agreement metric is Spearman correlation. For pairs involving DisProt, which is binary, the primary agreement metric is AUROC. These choices mirror the evaluation style used in the UdonPred 7x7 matrix: continuous test datasets use rank correlation, while DisProt uses binary ranking metrics.


In [ ]:
primary_metrics = {
    "continuous-continuous": "spearman",
    "continuous-binary": "auroc",
    "binary-continuous": "auroc",
    "binary-binary": "mcc",
}

primary_rows = []
for row in all_pairs.itertuples(index=False):
    pair_metrics = ceiling_summary[
        (ceiling_summary["dataset_a"] == row.dataset_a)
        & (ceiling_summary["dataset_b"] == row.dataset_b)
    ].copy()
    if pair_metrics.empty:
        continue
    type_key = f"{pair_metrics.iloc[0]['annotation_type_a']}-{pair_metrics.iloc[0]['annotation_type_b']}"
    metric = primary_metrics.get(type_key, "spearman")
    metric_row = pair_metrics[pair_metrics["metric"] == metric]
    if metric_row.empty:
        value = np.nan
    else:
        value = metric_row.iloc[0]["value"]
    primary_rows.append(
        {
            "dataset_a": row.dataset_a,
            "dataset_b": row.dataset_b,
            "metric": metric,
            "value": value,
            "n_residues_compared": row.n_residues_compared,
            "match_mode": row.match_mode,
        }
    )

primary_agreement = pd.DataFrame(primary_rows)
agreement_matrix = pd.DataFrame(np.nan, index=dataset_order, columns=dataset_order)
for row in primary_agreement.itertuples(index=False):
    agreement_matrix.loc[row.dataset_a, row.dataset_b] = row.value
    agreement_matrix.loc[row.dataset_b, row.dataset_a] = row.value

plt.figure(figsize=(8, 6))
sns.heatmap(
    agreement_matrix,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    vmin=-1,
    vmax=1,
    mask=agreement_matrix.isna(),
)
plt.title("Primary Annotation Agreement\nSpearman for continuous pairs, AUROC for DisProt pairs")
plt.show()

primary_agreement.sort_values("n_residues_compared", ascending=False)


## Compare UdonPred To The Estimated Ceiling

The table below compares UdonPred transfer scores with the primary annotation agreement for the same dataset pair where the metric is comparable. This is a rough diagnostic, not a strict mathematical bound, because the ceiling is estimated only on overlapping proteins/residues while the UdonPred matrix is evaluated on full test sets.

For pairs involving DisProt, only directions where DisProt is the test dataset are directly comparable to the AUROC ceiling metric.


In [ ]:
udon_matrix_path = RESULTS / "udonpred_matrix" / "matrix.csv"
if not udon_matrix_path.exists():
    raise FileNotFoundError(
        f"Missing {udon_matrix_path}. Run the UdonPred matrix notebook/script first."
    )

udon_matrix = pd.read_csv(udon_matrix_path).set_index("train_dataset")

def udon_column_for_test_dataset(test_dataset):
    if test_dataset == "disprot":
        return "disprot\n(AUROC)"
    return test_dataset

def udon_metric_for_test_dataset(test_dataset):
    if test_dataset == "disprot":
        return "auroc"
    return "spearman"

comparison_rows = []
for row in primary_agreement.itertuples(index=False):
    for train_dataset, test_dataset in [
        (row.dataset_a, row.dataset_b),
        (row.dataset_b, row.dataset_a),
    ]:
        test_metric = udon_metric_for_test_dataset(test_dataset)
        if test_metric != row.metric:
            continue
        column = udon_column_for_test_dataset(test_dataset)
        udon_value = udon_matrix.loc[train_dataset, column]
        comparison_rows.append(
            {
                "train_dataset": train_dataset,
                "test_dataset": test_dataset,
                "ceiling_pair": f"{row.dataset_a} vs {row.dataset_b}",
                "metric": row.metric,
                "udon_score": udon_value,
                "annotation_ceiling": row.value,
                "gap_to_ceiling": row.value - udon_value,
                "n_residues_compared": row.n_residues_compared,
                "match_mode": row.match_mode,
            }
        )

udon_vs_ceiling = pd.DataFrame(comparison_rows).sort_values(
    ["metric", "gap_to_ceiling"], ascending=[True, True]
)
udon_vs_ceiling


In [ ]:
metric_plot = ceiling_summary[
    ceiling_summary["metric"].isin(["spearman", "pearson", "auroc", "average_precision", "f1_thresholded"])
    & ceiling_summary["value"].notna()
].copy()
metric_plot["comparison"] = metric_plot["dataset_a"] + " vs " + metric_plot["dataset_b"]
metric_plot = metric_plot.sort_values(["metric", "value"], ascending=[True, False])

plt.figure(figsize=(12, 6))
sns.barplot(data=metric_plot, x="value", y="comparison", hue="metric")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Metric value")
plt.ylabel("Dataset pair")
plt.title("Annotation Agreement Metrics For Overlapping Dataset Pairs")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
ceiling_summary[
    ceiling_summary["notes"].notna() & (ceiling_summary["notes"] != "")
][["dataset_a", "dataset_b", "metric", "value", "notes"]].drop_duplicates().head(20)


## Full Interpretation of the Annotation Ceiling Results

### What the ceiling analysis means

The ceiling analysis estimates how much agreement is present in the annotations themselves before asking how well UdonPred can learn them. Following the lecture framing, this is essential because intrinsic disorder is not one single experimental observable. The slides emphasize that different disorder methods identify different proteins or regions, and that we should not assume that we already know one universal definition of disorder. In this project that is exactly the issue: TriZOD/CheZOD, SoftDis, PDBFlex, Atlas, pLDDT, and DisProt are all treated as disorder-related targets, but they do not measure the same biological quantity.

Therefore, a low ceiling is not automatically a model failure. It often means that two datasets encode different biological facets: NMR chemical-shift disorder, missing density or structural flexibility, predicted structural confidence, curated functional disorder, or a derived soft annotation. A model trained on one facet can only transfer cleanly to another facet when the underlying annotations agree on the same residues.

### Lecture-based biological framing

The lecture slides describe intrinsically disordered proteins and regions as proteins or regions that lack a fixed three-dimensional structure under physiological conditions, but still perform important biological roles in signaling, regulation, molecular recognition, and disease. The slides also stress that several historical prediction routes captured different proxies: long loop/no-regular-secondary-structure regions, B-values/flexibility, contact-deprived regions, curated disorder databases, and pLM-based disorder predictors. This explains why a ceiling analysis is biologically meaningful: disagreement between datasets can reflect real biological ambiguity rather than only noise.

For the datasets here, the interpretation is:

- `trizod` and `chezod` are closest to direct NMR chemical-shift disorder. They should be biologically related because both aim to capture local structural disorder from chemical shifts. TriZOD uses normalized G-scores, while CheZOD uses Z-score-like chemical-shift evidence, so agreement is expected but not perfect.
- `plddt` is not an experimental disorder measurement. It is AlphaFold confidence converted into a disorder-like score with `1 - pLDDT / 100`. Biologically, low confidence often tracks disorder, but it can also reflect missing evolutionary signal, modeling uncertainty, unresolved alternative conformations, or membrane/context effects.
- `disprot` is a curated binary annotation from heterogeneous evidence. It captures manually supported disorder, often with functional context, but it loses continuous residue-level nuance.
- `pdbflex` is related to B-factor/flexibility. The lecture explicitly warns that B-factors capture protein dynamics and flexibility, not disorder directly. This predicts weak ceilings against chemical-shift and pLDDT disorder.
- `softdis` and `atlas` are disorder-related derived resources, but their labels need not correspond one-to-one to NMR disorder, PDB flexibility, or curated binary disorder.

### Main numerical result

The strongest continuous-continuous agreement is `chezod` vs `plddt` with Spearman = 0.691 over 1,385 residues, followed by `softdis` vs `plddt` with Spearman = 0.657 over 2,521 residues. This suggests that AlphaFold confidence is aligned with a broad disorder signal in these overlapping examples. Biologically, this fits the lecture material on the high correlation between AlphaFold2 disorder-like uncertainty and pLM-based disorder predictors: many residues that are hard to fold confidently are also enriched for disorder-like sequence and structural context.

The next tier contains `trizod` vs `plddt` at Spearman = 0.505, `trizod` vs `chezod` at 0.482, and `trizod` vs `softdis` at 0.481. These are moderate, not high, ceilings. For NMR-based disorder this is important: even chemically grounded disorder annotations do not agree perfectly across datasets in the small overlaps available here. That supports the project motivation from the slides: data definition, not only model architecture, limits disorder prediction.

The weakest continuous-continuous agreements are centered on `pdbflex`: `pdbflex` vs `plddt` has Spearman = 0.012, `softdis` vs `pdbflex` has Spearman = 0.077 despite a very large overlap of 123,212 residues, and `pdbflex` vs `atlas` has Spearman = 0.111. This is the cleanest biological signal in the ceiling analysis. PDBFlex/B-factor-like flexibility is not equivalent to intrinsic disorder. A folded enzyme can have flexible loops, hinge regions, or crystal-packing-dependent mobility; an intrinsically disordered region can be absent from a structure or context-dependent. The lecture statement that B-factors capture aspects of dynamics, not directly disorder, is reflected almost exactly by these low ceilings.

### Pairwise biological interpretation

`chezod` vs `plddt` is the most convincing cross-source agreement. Because CheZOD is based on NMR chemical shifts and pLDDT is an AlphaFold confidence proxy, the high Spearman correlation suggests that many residues with experimentally disorder-like chemical shifts are also residues for which AlphaFold has low structural confidence. This supports using pLDDT as a useful disorder proxy, but not as a replacement for experiment, because the source of the signal is computational confidence rather than direct physical measurement.

`softdis` vs `plddt` also shows strong agreement. This suggests that SoftDis and AlphaFold uncertainty share a broad disorder-like signal, probably driven by sequence features of regions that are compositionally biased, contact-poor, or difficult to place into a stable fold. However, because both are partly prediction-derived or proxy-like compared with NMR, this agreement should not be overinterpreted as experimental confirmation.

`trizod` vs `chezod` is only moderate even though both are NMR-related. The overlap is just one protein and 138 residues, so confidence is low. Still, the moderate value is biologically plausible: TriZOD G-scores and CheZOD-style scores are related but not identical, and the project description notes that TriZOD was designed to avoid Z-score biases and improve normalization. The result says that NMR-derived annotations share a signal, but the measured ceiling is too overlap-limited to define a robust universal NMR ceiling.

`trizod` vs `softdis` is also moderate with six proteins and 563 residues. This means SoftDis captures part of the same residue ranking as TriZOD, but substantial disagreement remains. Biologically, this is what one would expect if SoftDis recognizes generic disorder-like sequence/structure properties while TriZOD measures local chemical-shift deviations more directly.

`softdis` vs `disprot` has AUROC = 0.930 over 2,642 residues, the strongest binary comparison. This indicates that SoftDis ranks DisProt-positive residues above DisProt-negative residues very well in the overlapping proteins. Biologically, this implies that SoftDis is close to the curated functional disorder concept represented by DisProt, at least for the small overlapping subset. It also explains why SoftDis-trained predictions can look good on DisProt AUROC even if SoftDis has only moderate correlations to some continuous datasets.

`pdbflex` vs `disprot` has AUROC = 0.308, and `pdbflex` vs `plddt` has Spearman = 0.012. These are not just weak; they suggest that PDBFlex labels are often measuring a different axis. This matches the lecture distinction between flexibility and disorder. High B-factors can occur in structured regions with thermal motion; curated DisProt disorder often marks regions lacking stable structure or undergoing disorder-to-order transitions. The biological conclusion is that flexibility-based targets should not be treated as interchangeable with IDR labels.

`softdis` vs `pdbflex` is especially informative because it has the largest overlap by far: 748 proteins and 123,212 residues, but Spearman is only 0.077. Unlike the tiny-overlap pairs, this low agreement is not mainly a sampling artifact. It strongly supports the interpretation that SoftDis-style disorder and PDBFlex-style flexibility encode distinct biological phenomena.

`atlas` has weak-to-moderate agreement with most other continuous sources: 0.315 with SoftDis, 0.295 with pLDDT, 0.221 with CheZOD, and 0.111 with PDBFlex. This places Atlas between broad disorder-like proxies and dataset-specific annotation behavior. Its biological meaning should therefore be interpreted cautiously: it may contain disorder signal, but it is not strongly interchangeable with any one source in the overlap analysis.

### What the gaps to UdonPred mean

The comparison between UdonPred scores and annotation ceilings is diagnostic, not a strict mathematical upper bound, because the ceiling is computed only on overlapping proteins/residues while UdonPred is evaluated on full test sets. Apparent scores above a ceiling do not necessarily mean that UdonPred exceeds experimental reproducibility; they often indicate that the overlap subset is too small or not representative. Examples include `atlas` to `plddt`, `pdbflex` to `plddt`, and `trizod` to `chezod`, where the model score on the full test set is higher than the estimated pairwise ceiling from limited overlap.

The reliable conclusion is qualitative: when the annotation ceiling between two datasets is low, cross-dataset transfer should not be expected to be robust. This is most important for PDBFlex. A model trained on PDBFlex is learning flexibility-like information and should not be expected to become a strong NMR-disorder or DisProt-disorder model. Conversely, `softdis`, `plddt`, `chezod`, and `trizod` share more disorder-like signal, so transfer among them is biologically more plausible.

### Confidence and limitations

Several ceiling estimates have very small overlap. `trizod` vs `chezod` has only one protein; `chezod` vs `softdis` also has one protein; `trizod` vs `plddt` has two proteins. These values should be read as examples of annotation compatibility, not stable population estimates. The most reliable negative conclusion is the weak `softdis` vs `pdbflex` agreement, because it is supported by a very large residue overlap.

No-overlap pairs are also meaningful for project planning. For example, `trizod` vs `disprot`, `trizod` vs `atlas`, `atlas` vs `disprot`, and `plddt` vs `disprot` have no exact ID or sequence overlap with enough comparable residues. For those pairs we cannot estimate an annotation ceiling from the current test sets. Any claims about biological agreement for these pairs must come indirectly from model transfer, literature, or future matched-protein analysis.

### Biological conclusion

The ceiling analysis supports the central biological message from the lecture slides and project description: protein disorder prediction is limited by heterogeneous annotation definitions. NMR chemical shifts, AlphaFold confidence, curated binary disorder, and structural flexibility all contain useful information, but they are not the same target. The results identify a shared disorder axis among CheZOD/TriZOD, SoftDis, pLDDT, and DisProt, while PDBFlex behaves as a distinct flexibility axis. This means that future modeling should not aim for one universal score without specifying the biological meaning of the target. A stronger approach is to report which disorder concept is being predicted and to use ceiling-aware interpretation when comparing datasets or choosing model heads.

